# Tests

> MCP tools for generating tests, stubs, and aggregating TODOs/BUGs

In [ ]:
#| default_exp tools.tests

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from typing import Any, Dict, List, Optional, Tuple

import re
import ast
import subprocess
from pathlib import Path

from mcp.server.fastmcp import FastMCP
from mcp.types import ToolAnnotations

from nbdev_mcp.utils.re import TODO_CODE_PATTERN, TODO_MD_PATTERN, BUG_PATTERN
from nbdev_mcp.utils.paths import (
    nbs_dir, settings_dict, resolve_selector, iter_notebooks,
    read_nb, write_nb, abs_module_for_nb, tutorials_dir,
)
from nbdev_mcp.utils.nb import join_source, find_default_exp, has_export
from nbdev_mcp.utils.rich import render_table, render_panel

## Tutorial & Error Scanning

In [ ]:
#| export
def run_tutorials(
    project: Optional[str] = None,
    timeout: Optional[int] = None,
    allow_errors: bool = False
) -> Dict[str, Any]:
    """Execute tutorial notebooks to verify they run without errors.
    
    Runs notebooks in tutorials/ directory using nbconvert or execnb.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias. Uses current project if not specified.
    timeout : int
        Timeout in seconds per notebook (default from config/environment).
    allow_errors : bool
        If True, continue even if a notebook fails.
    
    Returns
    -------
    Dict[str, Any]
        Result with 'results' list of per-notebook outcomes.
    """
    from nbdev_mcp.utils.config import get_config

    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}

    cfg = get_config()
    if timeout is None:
        timeout = cfg.timeout_run_tutorials
    
    tuts = tutorials_dir(p)
    if not tuts.exists():
        return {'ok': True, 'results': [], 'message': 'No tutorials/ directory found'}
    
    results: List[Dict[str, Any]] = []
    failed = 0
    
    for nb in sorted(tuts.glob('*.ipynb')):
        if '.ipynb_checkpoints' in str(nb):
            continue
        
        try:
            # Try using nbconvert to execute
            cmd = [
                'jupyter', 'nbconvert', '--to', 'notebook',
                '--execute', '--inplace',
                '--ExecutePreprocessor.timeout=' + str(timeout),
                str(nb)
            ]
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout + 60)
            ok = result.returncode == 0
            if not ok:
                failed += 1
            results.append({
                'notebook': str(nb.relative_to(p)),
                'ok': ok,
                'returncode': result.returncode,
                'stderr': result.stderr[:1000] if result.stderr else None
            })
        except subprocess.TimeoutExpired:
            failed += 1
            results.append({
                'notebook': str(nb.relative_to(p)),
                'ok': False,
                'error': f'Timeout after {timeout}s'
            })
        except FileNotFoundError:
            # jupyter not available
            return {'ok': False, 'error': 'jupyter nbconvert not found. Install jupyter to run tutorials.'}
        
        if not allow_errors and failed > 0:
            break
    
    rows = [[r['notebook'], '✓' if r['ok'] else '✗', r.get('error') or r.get('stderr', '')[:50] or '']
            for r in results]
    pretty = render_table('tutorial results', ['notebook', 'ok', 'notes'], rows)
    
    return {
        'ok': failed == 0,
        'results': results,
        'failed': failed,
        'total': len(results),
        'pretty': pretty
    }


In [ ]:
#| export
def scan_notebook_errors(
    project: Optional[str] = None,
    include_nbs: bool = True,
    include_tutorials: bool = True,
    max_trace_chars: int = 1200
) -> Dict[str, Any]:
    """Scan notebooks for existing error outputs without executing them.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias. Uses current project if not specified.
    include_nbs : bool
        If True, scan notebooks in nbs/ directory.
    include_tutorials : bool
        If True, scan notebooks in tutorials/ directory.
    max_trace_chars : int
        Maximum characters to include from traceback (default: 1200).
    
    Returns
    -------
    Dict[str, Any]
        Result with 'errors' list of notebooks with error outputs.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    errors: List[Dict[str, Any]] = []
    
    def scan_dir(d: Path) -> None:
        if not d.exists():
            return
        for nb in sorted(d.rglob('*.ipynb')):
            if '.ipynb_checkpoints' in str(nb):
                continue
            try:
                data = read_nb(nb)
            except Exception:
                continue
            
            for i, cell in enumerate(data.get('cells', [])):
                if cell.get('cell_type') != 'code':
                    continue
                for out in cell.get('outputs', []):
                    if out.get('output_type') == 'error':
                        ename = out.get('ename', 'Error')
                        evalue = out.get('evalue', '')
                        trace = '\n'.join(out.get('traceback', []))
                        errors.append({
                            'notebook': str(nb.relative_to(p)),
                            'cell': i,
                            'ename': ename,
                            'evalue': evalue[:200],
                            'traceback': trace[:max_trace_chars] if trace else None
                        })
    
    if include_nbs:
        scan_dir(nbs_dir(p))
    if include_tutorials:
        scan_dir(tutorials_dir(p))
    
    if errors:
        rows = [[e['notebook'], str(e['cell']), e['ename'], e['evalue'][:50]] for e in errors[:100]]
        pretty = render_table('notebook errors', ['notebook', 'cell', 'type', 'message'], rows)
    else:
        pretty = render_panel('scan_notebook_errors', 'No error outputs found in notebooks.')
    
    return {
        'ok': len(errors) == 0,
        'errors': errors,
        'total': len(errors),
        'pretty': pretty
    }


## Test Generation

In [ ]:
#| export
def generate_pytests(
    project: Optional[str] = None,
    dry_run: bool = False,
    long_test_eval_false: bool = True
) -> Dict[str, Any]:
    """Generate pytest-compatible modules from notebook test cells.
    
    Extracts test code from notebooks (cells with asserts or test functions)
    and generates pytest modules under tests/.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias. Uses current project if not specified.
    dry_run : bool
        If True, don't write files, just report what would be done.
    long_test_eval_false : bool
        If True, add '#| eval: false' to long test cells (40+ lines).
    
    Returns
    -------
    Dict[str, Any]
        Result with 'generated' list and 'modified_cells' list.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    s = settings_dict(p)
    lib = s.get('lib_name') or 'pkg'
    nbs_root = nbs_dir(p)
    outdir = p / 'tests'
    outdir.mkdir(exist_ok=True)
    
    generated: List[str] = []
    modified_cells: List[Tuple[str, int]] = []
    
    for nb in iter_notebooks(p):
        if not str(nb).startswith(str(nbs_root)) or nb.name == 'index.ipynb':
            continue
        
        data = read_nb(nb)
        mod = find_default_exp(data) or abs_module_for_nb(p, nb)[1]
        test_funcs: List[str] = []
        
        for i, cell in enumerate(data.get('cells', [])):
            if cell.get('cell_type') != 'code':
                continue
            src = join_source(cell.get('source', []))
            
            # Check if cell has test content
            if 'assert' not in src and not re.search(r'\bpytest\b|^def\s+test_', src, flags=re.MULTILINE):
                continue
            
            body_lines = [ln for ln in src.splitlines() if not ln.strip().startswith('#|')]
            if not body_lines:
                continue
            
            func_name = f'test_nb_cell_{i:03d}'
            test_funcs.append('\n'.join([
                f'def {func_name}():',
                *[f'    {ln}' if ln.strip() else '' for ln in body_lines],
                ''
            ]))
            
            # Mark long cells as eval: false
            if long_test_eval_false and len(body_lines) >= 40 and '#| eval: false' not in src:
                cell['source'] = ('#| eval: false\n' + src).splitlines(True)
                modified_cells.append((str(nb.relative_to(p)), i))
        
        if test_funcs:
            py = [
                f'# Auto-generated from {nb.relative_to(p)}\n',
                f'import {lib}.{mod} as m\n\n'
            ]
            py.extend(test_funcs)
            
            out_file = outdir / f"test_{mod.replace('.', '_')}.py"
            if not dry_run:
                out_file.write_text('\n'.join(py), encoding='utf-8')
            generated.append(str(out_file.relative_to(p)))
            
            if long_test_eval_false and modified_cells and not dry_run:
                write_nb(nb, data)
    
    if generated:
        rows = [[g] for g in generated]
        pretty = render_table('generated pytest files', ['path'], rows)
    else:
        pretty = render_panel('pytest generator', 'No tests found to extract.')
    
    return {
        'ok': True,
        'generated': generated,
        'modified_cells': modified_cells,
        'dry_run': dry_run,
        'pretty': pretty
    }

In [ ]:
#| export
def scaffold_test_utils(project: Optional[str] = None) -> Dict[str, Any]:
    """Create the standard nbs/99_tests/00_utils.ipynb notebook.
    
    Creates a testing utilities notebook with shared helpers and mocks
    for use across the test suite.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias. Uses current project if not specified.
    
    Returns
    -------
    Dict[str, Any]
        Result with 'created' path or 'message' if already exists.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    nb_dir = nbs_dir(p) / '99_tests'
    nb_dir.mkdir(parents=True, exist_ok=True)
    nb_path = nb_dir / '00_utils.ipynb'
    
    if nb_path.exists():
        return {'ok': True, 'message': f'Exists: {nb_path.relative_to(p)}'}
    
    nb = {
        'cells': [
            {
                'cell_type': 'markdown',
                'metadata': {},
                'source': ['# Test Utilities\n', '> Helpers and mocks for tests\n']
            },
            {
                'cell_type': 'code',
                'metadata': {},
                'source': ['#| default_exp tests.utils\n'],
                'outputs': [],
                'execution_count': None
            },
            {
                'cell_type': 'code',
                'metadata': {},
                'source': [
                    '#| export\n',
                    'def make_example_data(n: int = 3):\n',
                    '    "Return a simple list of ints for testing."\n',
                    '    return list(range(n))\n'
                ],
                'outputs': [],
                'execution_count': None
            }
        ],
        'metadata': {
            'kernelspec': {'name': 'python3', 'language': 'python'}
        },
        'nbformat': 4,
        'nbformat_minor': 5
    }
    
    write_nb(nb_path, nb)
    return {'ok': True, 'created': str(nb_path.relative_to(p))}

## Stub Generation

In [ ]:
#| export
def generate_stubs(
    project: Optional[str] = None,
    out_path: Optional[str] = None,
    include_methods: bool = True
) -> Dict[str, Any]:
    """Generate .pyi stub files for exported symbols.
    
    Scans notebooks for exported functions and classes, generating
    type stub files for better IDE support.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias. Uses current project if not specified.
    out_path : str, optional
        Custom output path for the stub file.
    include_methods : bool
        If True, include method signatures in class stubs.
    
    Returns
    -------
    Dict[str, Any]
        Result with 'out_file' path.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    s = settings_dict(p)
    lib = s.get('lib_name') or 'pkg'
    stubs_dir = p / 'stubs'
    stubs_dir.mkdir(exist_ok=True)
    out = Path(out_path) if out_path else stubs_dir / f'{lib}.pyi'
    
    modules: Dict[str, List[str]] = {}
    
    def add_stub(mod: str, text: str) -> None:
        modules.setdefault(mod, []).append(text)
    
    for nb in iter_notebooks(p):
        data = read_nb(nb)
        mod = find_default_exp(data) or abs_module_for_nb(p, nb)[1]
        
        for i, cell in enumerate(data.get('cells', [])):
            if cell.get('cell_type') != 'code':
                continue
            src = join_source(cell.get('source', []))
            if not has_export(src):
                continue
            
            try:
                tree = ast.parse(src)
            except SyntaxError:
                continue
            
            for node in tree.body:
                if isinstance(node, ast.FunctionDef):
                    sig = ast.get_source_segment(src, node.args) or '(…)'
                    add_stub(mod, f'def {node.name}{sig}: ...  # from {nb.name}#cell{i}')
                elif isinstance(node, ast.AsyncFunctionDef):
                    sig = ast.get_source_segment(src, node.args) or '(…)'
                    add_stub(mod, f'async def {node.name}{sig}: ...  # from {nb.name}#cell{i}')
                elif isinstance(node, ast.ClassDef):
                    bases = [ast.get_source_segment(src, b) or 'object' for b in node.bases]
                    add_stub(mod, f"class {node.name}({', '.join(bases) if bases else 'object'}): ...  # from {nb.name}#cell{i}")
                    if include_methods:
                        for sub in node.body:
                            if isinstance(sub, (ast.FunctionDef, ast.AsyncFunctionDef)):
                                sig = ast.get_source_segment(src, sub.args) or '(…)'
                                prefix = 'async def' if isinstance(sub, ast.AsyncFunctionDef) else 'def'
                                add_stub(mod, f'    {prefix} {sub.name}{sig}: ...')
    
    lines = [
        '# Auto-generated stubs — do not edit\n',
        f'# Package: {lib}\n\n',
        'from __future__ import annotations\n',
        'from typing import *\n\n'
    ]
    for mod in sorted(modules):
        lines.append(f'# --- {lib}.{mod} ---\n')
        for item in modules[mod]:
            lines.append(item + '\n')
        lines.append('\n')
    
    out.write_text(''.join(lines), encoding='utf-8')
    
    pretty = render_panel('stubs', f'Stub written: {out}')
    
    return {'ok': True, 'out_file': str(out.relative_to(p)), 'pretty': pretty}

## Comment Aggregation

In [ ]:
#| export
def aggregate_todos(
    project: Optional[str] = None,
    out_file: str = 'TODOs.md'
) -> Dict[str, Any]:
    """Aggregate TODO comments from notebooks into a Markdown file.
    
    Scans all notebooks for TODO comments and collects them into
    a single Markdown table.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias. Uses current project if not specified.
    out_file : str
        Output filename (default: TODOs.md).
    
    Returns
    -------
    Dict[str, Any]
        Result with 'count' of TODOs and 'out_file' path.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    rows: List[Tuple[str, int, int, str]] = []
    
    for nb in iter_notebooks(p):
        data = read_nb(nb)
        rel = str(nb.relative_to(p))
        
        for i, cell in enumerate(data.get('cells', [])):
            src = join_source(cell.get('source', []))
            
            if cell.get('cell_type') == 'code':
                for j, ln in enumerate(src.splitlines(), 1):
                    m = TODO_CODE_PATTERN.search(ln)
                    if m:
                        rows.append((rel, i, j, m.group(0).strip()))
            else:
                for j, ln in enumerate(src.splitlines(), 1):
                    if 'TODO' in ln.upper():
                        m = TODO_MD_PATTERN.search(ln)
                        if m:
                            rows.append((rel, i, j, m.group(0).strip()))
    
    outp = p / out_file
    lines = [
        '# TODOs\n\n',
        '| Notebook | Cell | Line | Comment |\n',
        '|---|---:|---:|---|\n'
    ]
    for rel, i, j, txt in rows:
        lines.append(f"| {rel} | {i} | {j} | {txt.replace('|', '\\|')} |\n")
    outp.write_text(''.join(lines), encoding='utf-8')
    
    return {'ok': True, 'count': len(rows), 'out_file': str(outp.relative_to(p))}

In [ ]:
#| export
def aggregate_bugs(
    project: Optional[str] = None,
    out_file: str = 'BUGs.md'
) -> Dict[str, Any]:
    """Aggregate BUG comments from notebooks into a Markdown file.
    
    Scans all notebooks for BUG comments and collects them into
    a single Markdown table.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias. Uses current project if not specified.
    out_file : str
        Output filename (default: BUGs.md).
    
    Returns
    -------
    Dict[str, Any]
        Result with 'count' of BUGs and 'out_file' path.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    rows: List[Tuple[str, int, int, str]] = []
    
    for nb in iter_notebooks(p):
        data = read_nb(nb)
        rel = str(nb.relative_to(p))
        
        for i, cell in enumerate(data.get('cells', [])):
            src = join_source(cell.get('source', []))
            for j, ln in enumerate(src.splitlines(), 1):
                if BUG_PATTERN.search(ln):
                    rows.append((rel, i, j, ln.strip()))
    
    outp = p / out_file
    lines = [
        '# BUGs\n\n',
        '| Notebook | Cell | Line | Comment |\n',
        '|---|---:|---:|---|\n'
    ]
    for rel, i, j, txt in rows:
        lines.append(f"| {rel} | {i} | {j} | {txt.replace('|', '\\|')} |\n")
    outp.write_text(''.join(lines), encoding='utf-8')
    
    return {'ok': True, 'count': len(rows), 'out_file': str(outp.relative_to(p))}

In [ ]:
#| export
def coverage_report(
    project: Optional[str] = None,
    include_untested: bool = True
) -> Dict[str, Any]:
    """Generate test coverage report per notebook.
    
    Analyzes which exported functions/classes have associated test
    cells (cells containing assert statements that reference them).
    
    Parameters
    ----------
    project : str, optional
        Project path or alias. Uses current project if not specified.
    include_untested : bool
        If True, list functions/classes without tests.
    
    Returns
    -------
    Dict[str, Any]
        Result with 'coverage' per notebook and 'untested' symbols.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    s = settings_dict(p)
    lib = s.get('lib_name') or 'pkg'
    
    # Track exported symbols per notebook
    exports: Dict[str, List[str]] = {}  # notebook -> [symbol names]
    tested: Dict[str, set] = {}  # notebook -> set of tested symbols
    
    for nb in iter_notebooks(p):
        data = read_nb(nb)
        rel = str(nb.relative_to(p))
        exports[rel] = []
        tested[rel] = set()
        
        for i, cell in enumerate(data.get('cells', [])):
            if cell.get('cell_type') != 'code':
                continue
            src = join_source(cell.get('source', []))
            
            # Find exported symbols
            if has_export(src):
                try:
                    tree = ast.parse(src)
                    for node in ast.walk(tree):
                        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                            exports[rel].append(node.name)
                        elif isinstance(node, ast.ClassDef):
                            exports[rel].append(node.name)
                except SyntaxError:
                    pass
            
            # Find test cells (cells with assert)
            if 'assert' in src:
                # Track which symbols are referenced in this test cell
                try:
                    tree = ast.parse(src)
                    for node in ast.walk(tree):
                        if isinstance(node, ast.Name):
                            tested[rel].add(node.id)
                        elif isinstance(node, ast.Call):
                            if isinstance(node.func, ast.Name):
                                tested[rel].add(node.func.id)
                            elif isinstance(node.func, ast.Attribute):
                                tested[rel].add(node.func.attr)
                except SyntaxError:
                    pass
    
    # Calculate coverage
    coverage: List[Dict[str, Any]] = []
    untested_list: List[Dict[str, str]] = []
    total_exports = 0
    total_tested = 0
    
    for nb, syms in exports.items():
        if not syms:
            continue
        
        nb_tested = [s for s in syms if s in tested[nb]]
        nb_untested = [s for s in syms if s not in tested[nb]]
        pct = (len(nb_tested) / len(syms) * 100) if syms else 0
        
        coverage.append({
            'notebook': nb,
            'exports': len(syms),
            'tested': len(nb_tested),
            'coverage': f'{pct:.0f}%'
        })
        
        total_exports += len(syms)
        total_tested += len(nb_tested)
        
        if include_untested:
            for s in nb_untested:
                untested_list.append({'notebook': nb, 'symbol': s})
    
    overall_pct = (total_tested / total_exports * 100) if total_exports else 0
    
    # Build output
    rows = [[c['notebook'], str(c['exports']), str(c['tested']), c['coverage']]
            for c in coverage]
    pretty = render_table('coverage report', ['notebook', 'exports', 'tested', 'coverage'], rows)
    pretty += f'\n\nOverall: {total_tested}/{total_exports} ({overall_pct:.0f}%)'
    
    if include_untested and untested_list:
        pretty += '\n\n' + render_panel('untested symbols', '\n'.join(
            f"- {u['notebook']}: {u['symbol']}" for u in untested_list[:50]
        ))
    
    return {
        'ok': True,
        'coverage': coverage,
        'untested': untested_list if include_untested else [],
        'total_exports': total_exports,
        'total_tested': total_tested,
        'overall_coverage': f'{overall_pct:.0f}%',
        'pretty': pretty
    }

## MCP Registration

In [ ]:
#| export
# Tool annotation definitions for test tools
TESTS_TOOL_ANNOTATIONS = {
    'run_tutorials': ToolAnnotations(
        title="Run Tutorials",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
    'scan_notebook_errors': ToolAnnotations(
        title="Scan Notebook Errors",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
    'generate_pytests': ToolAnnotations(
        title="Generate Pytests",
        readOnlyHint=False,  # Writes test files
        idempotentHint=True,
        openWorldHint=False
    ),
    'scaffold_test_utils': ToolAnnotations(
        title="Scaffold Test Utils",
        readOnlyHint=False,  # Creates notebook
        idempotentHint=True,
        openWorldHint=False
    ),
    'generate_stubs': ToolAnnotations(
        title="Generate Stubs",
        readOnlyHint=False,  # Writes .pyi files
        idempotentHint=True,
        openWorldHint=False
    ),
    'aggregate_todos': ToolAnnotations(
        title="Aggregate TODOs",
        readOnlyHint=False,  # Writes TODOs.md
        idempotentHint=True,
        openWorldHint=False
    ),
    'aggregate_bugs': ToolAnnotations(
        title="Aggregate BUGs",
        readOnlyHint=False,  # Writes BUGs.md
        idempotentHint=True,
        openWorldHint=False
    ),
    'coverage_report': ToolAnnotations(
        title="Coverage Report",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
}

def add_tests_tools(mcp: FastMCP) -> None:
    """Attach testing and generation tools to the MCP server with annotations.
    
    Parameters
    ----------
    mcp : FastMCP
        The MCP server instance.
    """
    mcp.add_tool(run_tutorials, annotations=TESTS_TOOL_ANNOTATIONS['run_tutorials'])
    mcp.add_tool(scan_notebook_errors, annotations=TESTS_TOOL_ANNOTATIONS['scan_notebook_errors'])
    mcp.add_tool(generate_pytests, annotations=TESTS_TOOL_ANNOTATIONS['generate_pytests'])
    mcp.add_tool(scaffold_test_utils, annotations=TESTS_TOOL_ANNOTATIONS['scaffold_test_utils'])
    mcp.add_tool(generate_stubs, annotations=TESTS_TOOL_ANNOTATIONS['generate_stubs'])
    mcp.add_tool(aggregate_todos, annotations=TESTS_TOOL_ANNOTATIONS['aggregate_todos'])
    mcp.add_tool(aggregate_bugs, annotations=TESTS_TOOL_ANNOTATIONS['aggregate_bugs'])
    mcp.add_tool(coverage_report, annotations=TESTS_TOOL_ANNOTATIONS['coverage_report'])

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()